In [ ]:
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

from scraping_beautifulsoup import scrape_books, scrape_gaaraas, scrape_motos

In [ ]:
url = "https://books.toscrape.com/catalogue/page-1.html"
response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

titres = []
for livre in soup.select("article.product_pod"):
    lien = livre.select_one("h3 a")
    if lien:
        titres.append(lien.get("title", lien.get_text(strip=True)))

titres[:5]

In [ ]:
limite_pages = 2
books = pd.DataFrame(scrape_books(limite_pages))
gaaraas = pd.DataFrame(scrape_gaaraas(limite_pages))
motos = pd.DataFrame(scrape_motos(limite_pages))

display(books.head())
display(gaaraas.head())
display(motos.head())

In [ ]:
def nettoyer(donnees):
    donnees = donnees.dropna(how="all").drop_duplicates().copy()
    donnees.columns = donnees.columns.str.strip()
    for colonne in donnees.select_dtypes(include="object"):
        donnees[colonne] = donnees[colonne].str.strip()
    return donnees


dossier = Path("exports")
dossier.mkdir(exist_ok=True)

for nom, donnees in [("books", books), ("gaaraas", gaaraas), ("motos", motos)]:
    donnees_nettoyees = nettoyer(donnees)
    donnees_nettoyees.to_csv(
        dossier / f"{nom}_nettoye.csv",
        index=False,
        encoding="utf-8-sig",
    )

display(nettoyer(books).head())